# R05 — The missing baseline ("filter the log, sort by recorded F1"), log-density sweep, and cost break-even

**CPU only, ~10–15 min (set `REVIEW_SMOKE=1` for a 1-minute plumbing check), no images needed.** Addresses
**R2.8** (simple baseline that filters the HPO log for the query and ranks by *recorded* F1),
**R1.6** (how retrieval degrades as the log gets sparser; what "adequate coverage" means),
**R1.7** (when is the framework cheaper than sequential retraining; is ACFX time acceptable).

Everything is computed from the committed HPO logs, saved surrogates and timing files. No PatchCore is retrained here:
the *recorded* F1 of a logged trial is the quantity the baseline ranks by. Whether recorded F1 survives retraining is
the subject of R02.

Retrieval rules compared (query suite = the 12 queries of `shared/07`, anchors p50 / p85 / best):

| rule | feasibility test | ranking |
|---|---|---|
| **RecordedTopK** (new baseline) | recorded F1 in band ∧ soft constraints | recorded F1, descending |
| RecordedNearest | same | mixed-type distance to the anchor |
| LogTopK (paper) | *surrogate* F1 in band ∧ soft constraints | surrogate F1, descending |
| NearestFeasible (paper) | *surrogate* F1 in band ∧ soft constraints | distance to the anchor |

In [ ]:
# ============================================================
# C0) Common setup (identical in every review notebook)
# ============================================================
from pathlib import Path
import json, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
try:
    display
except NameError:
    display = print

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 150, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})
SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)

# ------------------------------------------------------------
# Robust repo-root detection (CWD, its parents, then rglob)
# ------------------------------------------------------------
CWD = Path.cwd().resolve()


def _is_repo(p: Path) -> bool:
    return (p / "mvtec" / "outputs_patchcore").is_dir() and (p / "proditec" / "outputs_patchcore").is_dir()


REPO = next((p for p in [CWD, *CWD.parents] if _is_repo(p)), None)
if REPO is None:
    for base in [CWD, CWD.parent]:
        for hit in base.rglob("outputs_patchcore"):
            if _is_repo(hit.parent.parent):
                REPO = hit.parent.parent
                break
        if REPO:
            break
if REPO is None:
    raise FileNotFoundError("Could not locate the repository root (needs mvtec/ and proditec/). "
                            "Launch Jupyter from the repo or from review/.")
REVIEW = REPO / "review"
OUT = REVIEW / "outputs"
OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
for d in (OUT_TABLES, OUT_FIGS):
    d.mkdir(parents=True, exist_ok=True)

DS = {
    "mvtec": dict(root=REPO / "mvtec", pre="mvtec",
                  data="surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv"),
    "proditec": dict(root=REPO / "proditec", pre="proditec",
                     data="surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"),
}
for k, v in DS.items():
    r = v["root"]
    v["art"] = r / "surrogate_pkl_cfs_metadata"
    v["cfs"] = v["art"] / f"{k}_counterfactuals" / f"{k}_all_methods_cfs.csv"
    v["eval"] = v["art"] / f"{k}_counterfactuals_evaluation" / f"{k}_all_methods_cfs_patchcore_evaluation.csv"
    v["rev"] = v["art"] / f"{k}_revision_analysis"
    v["decomp"] = r / "revision_outputs" / "tables" / f"decomposition_per_candidate_{k}.csv"
    v["model"] = v["art"] / f"{k}_surrogate_catboost.cbm"
    v["meta"] = v["art"] / f"{k}_surrogate_metadata.json"
    v["dataset"] = r / v["data"]
    v["confirm"] = r / "revision_outputs" / "tables" / f"baseline_confirmation_{k}.csv"
    v["confirm_cands"] = r / "revision_outputs" / "tables" / f"baseline_confirm_candidates_{k}.csv"
print("REPO  :", REPO)
print("OUT   :", OUT)


def save_table(table: pd.DataFrame, stem: str, ff="%.4f"):
    p = OUT_TABLES / f"{stem}.csv"
    table.to_csv(p, index=False)
    try:
        table.to_latex(OUT_TABLES / f"{stem}.tex", index=False, escape=True, float_format=ff)
    except Exception as e:
        print("  (tex export skipped:", e, ")")
    print("Saved table:", p.relative_to(REPO))
    return p


def save_fig(fig, stem: str):
    for ext in ("pdf", "png"):
        fig.savefig(OUT_FIGS / f"{stem}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("Saved figure:", (OUT_FIGS / f"{stem}.pdf").relative_to(REPO))


def merge_numbers(section: str, values: dict):
    """Append to outputs/frozen_numbers.json (one section per notebook)."""
    p = OUT / "frozen_numbers.json"
    cur = json.loads(p.read_text()) if p.exists() else {}
    cur[section] = json.loads(json.dumps(values, default=float))
    p.write_text(json.dumps(cur, indent=2, sort_keys=True))
    print("Updated", p.relative_to(REPO), "->", section)

In [ ]:
# ============================================================
# R5.1) Setup: logs, surrogates, query machinery (copied from shared/07 S7.1-S7.2, retrieval part only)
# ============================================================
import os, time, io, contextlib
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

QUICK = os.environ.get("REVIEW_SMOKE", "0") == "1"
TARGET = "value"
K = 8
ANCHOR_QUANTILES = [0.50, 0.85]            # + best trial
DENSITY_FRACTIONS = [0.25] if QUICK else [0.50, 0.25, 0.10, 0.05]
DENSITY_REPS = 1 if QUICK else 3
RUN_DICE_DENSITY = True                    # generation on the thinned log (needs dice-ml); set False to skip

CTX = {}
for ds, s in DS.items():
    meta = json.loads(s["meta"].read_text())
    feats, cats = meta["features"], meta["categorical_features"]
    nums = [c for c in feats if c not in cats]
    df = pd.read_csv(s["dataset"])[[TARGET] + feats].copy()
    for c in cats:
        df[c] = df[c].astype(str)
    for c in nums + [TARGET]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna().reset_index(drop=True)
    cb = CatBoostRegressor(); cb.load_model(str(s["model"]))
    cols = list(cb.feature_names_)

    def coerce(frame, cats=cats, nums=nums):
        x = frame.copy()
        for c in cats:
            if c in x.columns: x[c] = x[c].astype(str)
        for c in nums:
            if c in x.columns: x[c] = pd.to_numeric(x[c], errors="coerce")
        return x

    CTX[ds] = dict(ds=ds, df=df, FEATURES=feats, CATEGORICAL=cats, NUMERIC=nums, coerce=coerce,
                   predict=(lambda X, cb=cb, cols=cols, coerce=coerce: np.asarray(cb.predict(coerce(X[cols])), float)),
                   range_df=df, model=cb, cols=cols)
    CTX[ds]["pred_log"] = CTX[ds]["predict"](df)
    print(f"{ds}: log={df.shape}, recorded F1 range [{df[TARGET].min():.3f}, {df[TARGET].max():.3f}], "
          f"surrogate-vs-recorded R2 on the whole log = "
          f"{1 - ((df[TARGET] - CTX[ds]['pred_log']) ** 2).sum() / ((df[TARGET] - df[TARGET].mean()) ** 2).sum():.4f}")


def mixed_distance(ctx, cand_df, anchor_row):
    d = np.zeros(len(cand_df), dtype=float)
    for c in ctx["NUMERIC"]:
        r = float(ctx["range_df"][c].max() - ctx["range_df"][c].min()) or 1.0
        d += np.abs(pd.to_numeric(cand_df[c], errors="coerce").to_numpy() - float(anchor_row[c])) / r
    for c in ctx["CATEGORICAL"]:
        d += (cand_df[c].astype(str).to_numpy() != str(anchor_row[c])).astype(float)
    return d


def resolve_query(q, ctx, anchor_row, anchor_val):
    ob = q["objective"]
    if ob[0] == "target_band":      L, U = ob[1], ob[2]
    elif ob[0] == "target_band_q":  L, U = (float(ctx["range_df"][TARGET].quantile(ob[1])), float(ctx["range_df"][TARGET].quantile(ob[2])))
    elif ob[0] == "target_min":     L, U = ob[1], 1.0
    elif ob[0] == "delta_improve":  L, U = anchor_val + ob[1], 1.0
    tf, rb, cr = q.get("train_fraction", ("free",)), q.get("review_budget", ("free",)), q.get("corruption", ("free",))

    def rng(spec, lo_all, hi_all, anchor_v=None):
        if spec[0] == "free":    return (lo_all, hi_all)
        if spec[0] == "range":   return (max(lo_all, spec[1]), min(hi_all, spec[2]))
        if spec[0] == "max":     return (lo_all, min(hi_all, spec[1]))
        if spec[0] == "rel_max": return (lo_all, min(hi_all, spec[1] * float(anchor_v)))
        if spec[0] == "fixed":   return (spec[1], spec[1])
        raise ValueError(spec)
    tf_b = rng(tf, 0.20, 1.00, anchor_row["params_soft_train_fraction"])
    rb_b = rng(rb, 0.005, 0.500, anchor_row["params_soft_review_budget"])
    cr_s = {"none", "mild", "strong"} if cr[0] == "free" else (set(cr[1]) if cr[0] == "allowed" else {cr[1]})
    return dict(L=L, U=U, tf=tf_b, rb=rb_b, corr=cr_s)


def feasible_mask(frame, rq):
    tf = pd.to_numeric(frame["params_soft_train_fraction"], errors="coerce")
    rb = pd.to_numeric(frame["params_soft_review_budget"], errors="coerce")
    cl = frame["params_soft_corruption_level"].astype(str)
    return ((tf >= rq["tf"][0] - 1e-9) & (tf <= rq["tf"][1] + 1e-9) & (rb >= rq["rb"][0] - 1e-9)
            & (rb <= rq["rb"][1] + 1e-9) & cl.isin(rq["corr"]))


QUERY_SUITE = [
    dict(qid="broad_band_free", objective=("target_band", 0.75, 0.95)),
    dict(qid="tight_band", objective=("target_band", 0.85, 0.90)),
    dict(qid="target_min_085", objective=("target_min", 0.85)),
    dict(qid="delta_improve_005", objective=("delta_improve", 0.05)),
    dict(qid="low_budget", objective=("target_band", 0.75, 0.95), review_budget=("max", 0.30)),
    dict(qid="low_train_fraction", objective=("target_band", 0.75, 0.95), train_fraction=("max", 0.60)),
    dict(qid="corruption_none_mild", objective=("target_band", 0.75, 0.95), corruption=("allowed", ["none", "mild"])),
    dict(qid="relative_budget_090", objective=("delta_improve", 0.0), review_budget=("rel_max", 0.90)),
    dict(qid="combo_tight", objective=("target_band", 0.85, 0.92), review_budget=("max", 0.30),
         train_fraction=("max", 0.60), corruption=("allowed", ["none", "mild"])),
    dict(qid="infeasible_high", objective=("target_min", 0.99)),
    dict(qid="tight_band_q", objective=("target_band_q", 0.80, 0.95)),
    dict(qid="combo_q", objective=("target_band_q", 0.80, 0.95), review_budget=("max", 0.30),
         train_fraction=("max", 0.60), corruption=("allowed", ["none", "mild"])),
]


def make_anchors(ctx):
    df = ctx["df"]; out = []
    idxs = [(f"p{int(q * 100)}", int((df[TARGET] - df[TARGET].quantile(q)).abs().idxmin())) for q in ANCHOR_QUANTILES]
    idxs.append(("best", int(df[TARGET].idxmax())))
    for lab, i in idxs:
        out.append(dict(label=lab, idx=i, row=df.loc[i, ctx["FEATURES"]], recorded=float(df.loc[i, TARGET]),
                        pred=float(ctx["pred_log"][i])))
    return out


for ds, ctx in CTX.items():
    ctx["anchors"] = make_anchors(ctx)
    print(ds, [(a["label"], round(a["recorded"], 3), round(a["pred"], 3)) for a in ctx["anchors"]])

In [ ]:
# ============================================================
# R5.2) Four retrieval rules on the full log
# ============================================================
def dedupe(frame, order_idx, ctx, k):
    sig = frame.iloc[order_idx][ctx["FEATURES"]].astype(str).agg("|".join, axis=1)
    keep = ~sig.duplicated()
    return np.asarray(order_idx)[keep.to_numpy()][:k]


def retrieve(ctx, df, pred_df, rule, rq, anchor, k=K):
    """Return row positions (in ``df``) chosen by ``rule``."""
    rec, pred = df[TARGET].to_numpy(), pred_df
    fm = feasible_mask(df, rq).to_numpy()
    if rule.startswith("Recorded"):
        ok = fm & (rec >= rq["L_rec"]) & (rec <= rq["U"])
        score = rec
    else:
        ok = fm & (pred >= rq["L"]) & (pred <= rq["U"])
        score = pred
    idx = np.where(ok)[0]
    if len(idx) == 0:
        return idx, int(fm.sum()), 0
    if rule in ("RecordedTopK", "LogTopK"):
        order = idx[np.argsort(-score[idx], kind="stable")]
    else:
        d = mixed_distance(ctx, df.iloc[idx], anchor["row"])
        order = idx[np.argsort(d, kind="stable")]
    return dedupe(df, order, ctx, k), int(fm.sum()), int(len(idx))


RULES = ["RecordedTopK", "RecordedNearest", "LogTopK", "NearestFeasible"]
rows, picks = [], {}
for ds, ctx in CTX.items():
    df, pred = ctx["df"], ctx["pred_log"]
    for q in QUERY_SUITE:
        for a in ctx["anchors"]:
            rq = resolve_query(q, ctx, a["row"], a["pred"])                     # surrogate-scale bounds
            rq["L_rec"] = resolve_query(q, ctx, a["row"], a["recorded"])["L"]   # recorded-scale lower bound
            for rule in RULES:
                idx, n_feas, n_band = retrieve(ctx, df, pred, rule, rq, a)
                picks[(ds, q["qid"], a["label"], rule)] = idx
                if len(idx):
                    rec, pr = df[TARGET].to_numpy()[idx], pred[idx]
                    dist = mixed_distance(ctx, df.iloc[idx], a["row"])
                    in_rec = ((rec >= rq["L_rec"]) & (rec <= rq["U"])).mean()
                    in_sur = ((pr >= rq["L"]) & (pr <= rq["U"])).mean()
                    rows.append(dict(dataset=ds, qid=q["qid"], anchor=a["label"], rule=rule, n_returned=len(idx), shortfall=K - len(idx),
                                     feasible_soft=n_feas, in_band_candidates=n_band, mean_recorded=rec.mean(), mean_surrogate=pr.mean(),
                                     mean_abs_surrogate_error=np.abs(pr - rec).mean(), mean_distance=dist.mean(),
                                     frac_recorded_in_band=in_rec, frac_surrogate_in_band=in_sur))
                else:
                    rows.append(dict(dataset=ds, qid=q["qid"], anchor=a["label"], rule=rule, n_returned=0, shortfall=K,
                                     feasible_soft=n_feas, in_band_candidates=n_band))
R = pd.DataFrame(rows)
save_table(R.round(4), "R05_retrieval_rules_per_cell")

agg = (R[R.n_returned > 0].groupby(["dataset", "rule"])
       .agg(cells_answered=("qid", "size"), mean_returned=("n_returned", "mean"), mean_recorded=("mean_recorded", "mean"),
            mean_surrogate=("mean_surrogate", "mean"), mean_abs_err=("mean_abs_surrogate_error", "mean"),
            mean_distance=("mean_distance", "mean"), frac_recorded_in_band=("frac_recorded_in_band", "mean"),
            frac_surrogate_in_band=("frac_surrogate_in_band", "mean")).reset_index())
total = R.groupby(["dataset", "rule"]).size().rename("cells_total").reset_index()
agg = agg.merge(total, on=["dataset", "rule"])
display(agg.round(4))
save_table(agg.round(4), "R05_retrieval_rules_aggregate")

**Reading the table.** `frac_recorded_in_band` is the fraction of returned candidates whose *recorded* F1 satisfies the
query. The recorded-F1 baselines satisfy it by construction; for LogTopK/NearestFeasible it measures how often the
surrogate-ranked pick would have been rejected by a plain filter on the log. `mean_recorded` is what the operator
would be promised by the log itself; `mean_abs_err` is the surrogate-vs-recorded discrepancy of the returned items
(selection-conditioned fidelity measured directly on the log, without retraining).

In [ ]:
# ============================================================
# R5.3) Head-to-head: does surrogate ranking add anything over ranking by recorded F1?
# ============================================================
rows = []
for ds, ctx in CTX.items():
    df = ctx["df"]
    for q in QUERY_SUITE:
        for a in ctx["anchors"]:
            ia, ib = picks[(ds, q["qid"], a["label"], "RecordedTopK")], picks[(ds, q["qid"], a["label"], "LogTopK")]
            if len(ia) == 0 and len(ib) == 0:
                continue
            sa, sb = set(df.iloc[ia][ctx["FEATURES"]].astype(str).agg("|".join, axis=1)), set(df.iloc[ib][ctx["FEATURES"]].astype(str).agg("|".join, axis=1))
            rows.append(dict(dataset=ds, qid=q["qid"], anchor=a["label"], jaccard_recordedtopk_vs_logtopk=len(sa & sb) / max(1, len(sa | sb)),
                             recorded_mean_recordedtopk=df[TARGET].to_numpy()[ia].mean() if len(ia) else np.nan,
                             recorded_mean_logtopk=df[TARGET].to_numpy()[ib].mean() if len(ib) else np.nan))
H = pd.DataFrame(rows)
H["recorded_gain_of_surrogate_rank"] = H.recorded_mean_logtopk - H.recorded_mean_recordedtopk
display(H.groupby("dataset").agg(cells=("qid", "size"), mean_jaccard=("jaccard_recordedtopk_vs_logtopk", "mean"),
                                 mean_recorded_gain_of_surrogate_rank=("recorded_gain_of_surrogate_rank", "mean"),
                                 frac_cells_surrogate_rank_worse_on_recorded=("recorded_gain_of_surrogate_rank", lambda s: float((s < -1e-9).mean()))).round(4))
save_table(H.round(4), "R05_recordedtopk_vs_logtopk")

In [ ]:
# ============================================================
# R5.4) Where could generation matter at all? Cells in which NO logged trial satisfies the query
#       (using recorded F1) but the paper's generators produced surrogate-valid candidates.
# ============================================================
qs = []
for ds, s in DS.items():
    p = s["root"] / "revision_outputs" / "tables" / f"query_suite_{ds}.csv"
    if p.exists():
        qs.append(pd.read_csv(p))
if qs:
    QS = pd.concat(qs, ignore_index=True)
    rec = R[R.rule == "RecordedTopK"][["dataset", "qid", "anchor", "n_returned", "in_band_candidates"]].rename(
        columns={"n_returned": "recorded_baseline_n", "in_band_candidates": "recorded_in_band_candidates"})
    dice = QS[QS.method == "DiCE"][["dataset", "qid", "anchor", "final_valid_n", "raw_valid_n"]].rename(
        columns={"final_valid_n": "dice_final_valid_n", "raw_valid_n": "dice_raw_valid_n"})
    G = dice.merge(rec, on=["dataset", "qid", "anchor"], how="left")
    G["dice_answers_but_recorded_baseline_does_not"] = (G.dice_final_valid_n >= 1) & (G.recorded_baseline_n == 0)
    G["baseline_answers_but_dice_shortfall"] = (G.recorded_baseline_n >= 1) & (G.dice_final_valid_n < K)
    display(G.groupby("dataset").agg(cells=("qid", "size"),
                                     dice_only=("dice_answers_but_recorded_baseline_does_not", "sum"),
                                     baseline_answers_dice_short=("baseline_answers_but_dice_shortfall", "sum")))
    display(G[G.dice_answers_but_recorded_baseline_does_not])
    save_table(G.round(4), "R05_generation_vs_recorded_baseline_coverage")
else:
    print("query_suite_*.csv not found; skipping")

In [ ]:
# ============================================================
# R5.5) Log-density sweep (R1.6): thin the log, REFIT the surrogate on what remains, rerun the suite.
#   metrics: shortfall of retrieval (LogTopK, RecordedTopK) and, optionally, of DiCE (surrogate-level, post-snapping)
#   Domain bounds for DiCE stay those of the full search space (they are declared, not learned from the log).
# ============================================================
def thin_context(ctx, frac, seed):
    df = ctx["df"]
    sub = df.sample(frac=frac, random_state=seed).reset_index(drop=True)
    feats, cats = ctx["FEATURES"], ctx["CATEGORICAL"]
    Xa, Xv, ya, yv = train_test_split(sub[feats], sub[TARGET], test_size=0.1, random_state=seed)
    m = CatBoostRegressor(loss_function="RMSE", iterations=3000, learning_rate=0.05, depth=8, l2_leaf_reg=3.0,
                          random_seed=seed, verbose=False, thread_count=-1)
    m.fit(Xa, ya, cat_features=[feats.index(c) for c in cats], eval_set=(Xv, yv), use_best_model=True, early_stopping_rounds=100)
    cols = list(m.feature_names_)
    new = dict(ctx)
    new.update(df=sub, model=m, predict=(lambda X, m=m, cols=cols, coerce=ctx["coerce"]: np.asarray(m.predict(coerce(X[cols])), float)))
    new["pred_log"] = new["predict"](sub)
    return new


def make_dice(ctx):
    import dice_ml
    from dice_ml import Dice
    dd = ctx["df"][ctx["FEATURES"] + [TARGET]].copy()
    cont = [c for c in ctx["FEATURES"] if c not in ctx["CATEGORICAL"]]
    data = dice_ml.Data(dataframe=dd, continuous_features=cont, outcome_name=TARGET)

    class Reg:
        def __init__(self, f): self.f = f
        def predict(self, X): return self.f(X)
    return Dice(data, dice_ml.Model(model=Reg(ctx["predict"]), backend="sklearn", model_type="regressor"), method="random")


_NUM_SNAP = {"params_soft_review_budget": (0.005, 0.0), "params_soft_train_fraction": (0.01, 0.0),
             "params_coreset_sampling_ratio": (0.002, 0.001), "params_num_neighbors": (1.0, 0.0)}


def snap(ctx, x):
    x = ctx["coerce"](x.copy())
    for c, (st, o) in _NUM_SNAP.items():
        if c in x.columns:
            x[c] = o + np.round((pd.to_numeric(x[c], errors="coerce") - o) / st) * st
    return x


def dice_valid_n(ctx_thin, ctx_full, dice, rq, anchor, k=K):
    x_df = ctx_thin["coerce"](pd.DataFrame([anchor["row"].values], columns=ctx_thin["FEATURES"]))
    pr = {}
    for c in ctx_full["NUMERIC"]:
        lo, hi = float(ctx_full["range_df"][c].min()), float(ctx_full["range_df"][c].max())
        if c == "params_soft_train_fraction": lo, hi = rq["tf"]
        if c == "params_soft_review_budget": lo, hi = rq["rb"]
        pr[c] = [lo, hi]
    for c in ctx_full["CATEGORICAL"]:
        pr[c] = sorted(ctx_full["range_df"][c].astype(str).unique().tolist())
    pr["params_soft_corruption_level"] = sorted(rq["corr"])
    try:
        with contextlib.redirect_stdout(io.StringIO()):     # DiCE prints a line for every infeasible query
            res = dice.generate_counterfactuals(x_df[ctx_thin["FEATURES"]], total_CFs=max(30, 5 * k), desired_range=[rq["L"], rq["U"]],
                                                features_to_vary=ctx_thin["FEATURES"], permitted_range=pr)
        ex = res.cf_examples_list[0]
        raw = (ex.final_cfs_df_sparse if getattr(ex, "final_cfs_df_sparse", None) is not None else ex.final_cfs_df).copy()
        x = snap(ctx_thin, raw.reindex(columns=ctx_thin["FEATURES"]).dropna())
        p = ctx_thin["predict"](x)
        ok = (p >= rq["L"]) & (p <= rq["U"]) & feasible_mask(x, rq).to_numpy()
        sig = x[ok].astype(str).agg("|".join, axis=1)
        return int((~sig.duplicated()).sum()), None
    except Exception as e:
        return 0, repr(e)[:120]


rows = []
t0 = time.time()
for ds, ctx in CTX.items():
    for frac in [1.0] + DENSITY_FRACTIONS:
        for rep in range(1 if frac == 1.0 else DENSITY_REPS):
            c2 = ctx if frac == 1.0 else thin_context(ctx, frac, SEED + rep)
            dice = make_dice(c2) if RUN_DICE_DENSITY else None
            for q in QUERY_SUITE:
                for a in ctx["anchors"]:
                    rq = resolve_query(q, ctx, a["row"], a["pred"])
                    rq["L_rec"] = resolve_query(q, ctx, a["row"], a["recorded"])["L"]
                    out = dict(dataset=ds, fraction=frac, rep=rep, n_log=len(c2["df"]), qid=q["qid"], anchor=a["label"])
                    for rule in ("RecordedTopK", "LogTopK"):
                        idx, n_feas, n_band = retrieve(c2, c2["df"], c2["pred_log"], rule, rq, a)
                        out[f"{rule}_n"] = len(idx)
                    if dice is not None:
                        out["DiCE_n"], out["DiCE_err"] = dice_valid_n(c2, ctx, dice, rq, a)
                    rows.append(out)
            print(f"{ds} frac={frac} rep={rep} done ({time.time() - t0:.0f}s)")
DEN = pd.DataFrame(rows)
save_table(DEN, "R05_density_sweep_per_cell")

cols = [c for c in ["RecordedTopK_n", "LogTopK_n", "DiCE_n"] if c in DEN.columns]
# only cells that are answerable on the FULL log (otherwise 0 is the right answer)
full = DEN[DEN.fraction == 1.0].set_index(["dataset", "qid", "anchor"])
answerable = full[full["RecordedTopK_n"] >= 1].index
D2 = DEN.set_index(["dataset", "qid", "anchor"]).loc[answerable].reset_index()
summary = []
for (ds, frac), g in D2.groupby(["dataset", "fraction"]):
    row = dict(dataset=ds, fraction=frac, n_log=int(g.n_log.mean()), answerable_cells=int(g.groupby(["qid", "anchor"]).ngroups))
    for c in cols:
        row[f"{c[:-2]}_mean_returned_of_{K}"] = g[c].mean()
        row[f"{c[:-2]}_frac_cells_full_k"] = float((g[c] >= K).mean())
        row[f"{c[:-2]}_frac_cells_any"] = float((g[c] >= 1).mean())
    summary.append(row)
SUM = pd.DataFrame(summary)
display(SUM.round(3))
save_table(SUM.round(4), "R05_density_sweep_summary")

fig, axes = plt.subplots(1, 2, figsize=WIDE_COL, sharey=True)
for ax, ds in zip(axes, DS):
    t = SUM[SUM.dataset == ds].sort_values("n_log")
    for c, col, ls in [("RecordedTopK", "k", "-"), ("LogTopK", "#3b82d4", "--"), ("DiCE", "#7c5cd8", ":")]:
        if f"{c}_frac_cells_full_k" in t:
            ax.plot(t.n_log, t[f"{c}_frac_cells_full_k"], color=col, ls=ls, marker="o", ms=3, label=c)
    ax.set_xscale("log"); ax.set_title(ds); ax.set_xlabel("number of logged trials available")
axes[0].set_ylabel(f"fraction of answerable cells with all {K} alternatives"); axes[0].legend(frameon=False)
save_fig(fig, "R05_density_sweep")

**Operational reading.** The sweep gives a concrete "adequate coverage" rule: the number of logged trials at which retrieval
still returns all K alternatives in a given fraction of the answerable queries. Combine with the thinned-log surrogate
refit (R² on the thinned log is reported in the per-cell file via `n_log`) and with the relevant constraint
selectivity (`feasible_soft`, `in_band_candidates` in `R05_retrieval_rules_per_cell.csv`).

In [ ]:
# ============================================================
# R5.6) Cost model and break-even (R1.7)
#   inputs: measured PatchCore retraining time (eval CSVs), GPU/CPU language-model stage times, generation times,
#           HPO log creation cost (per-trial runtime stored in the Optuna DBs)
# ============================================================
sys.path.insert(0, str(REVIEW))
import patchcore_review_lib as L   # only for load_hpo_trials (no GPU needed)

lat = json.loads((REPO / "shared" / "revision_outputs" / "parser_eval_latency.json").read_text())
cpu = pd.read_csv(DS["mvtec"]["root"] / "revision_outputs" / "tables" / "end_to_end_latency_mvtec.csv")
LM = {  # seconds
    "CPU (as measured in the paper)": dict(parse=float(cpu.parse_s.iloc[0]), explain=float(cpu.explain_s.iloc[0])),
    "GPU bf16 (parser_eval_latency.json)": dict(parse=lat["parse_mean_s_as_deployed"], explain=lat["explain_mean_s"]),
}
GEN = {}
for ds, s in DS.items():
    e = pd.read_csv(s["eval"]); c = pd.read_csv(s["cfs"])
    GEN[ds] = dict(retrain_mean=float(e.runtime_sec.mean()), retrain_median=float(e.runtime_sec.median()),
                   ACFX=float(c[c.method == "ACFX"].generation_sec.mean()), CFNOW=float(c[c.method == "CFNOW"].generation_sec.mean()),
                   DiCE=float(c[c.method == "DiCE"].generation_sec.mean()), LogTopK=0.0014, NearestFeasible=0.0019)
hpo = {"mvtec": L.load_hpo_trials(DS["mvtec"]["root"] / "outputs_patchcore" / "mvtec_4class_patchcore_hpo_logs.db"),
       "proditec": L.load_hpo_trials(DS["proditec"]["root"] / "outputs_patchcore" / "pdt_public_patchcore_hpo_logs.db", "v1.0.1")}
for ds in DS:
    GEN[ds]["hpo_trials"] = int(len(hpo[ds])); GEN[ds]["hpo_trial_mean_s"] = float(hpo[ds].runtime_sec.mean())
    GEN[ds]["hpo_total_gpu_h"] = float(hpo[ds].runtime_sec.sum() / 3600)
print(json.dumps(GEN, indent=1))
print("Paper Table 7 quotes retraining 30.8 s (MVTec) / 67.1 s (Proditec); the committed confirmation files give "
      f"{GEN['mvtec']['retrain_mean']:.1f} s / {GEN['proditec']['retrain_mean']:.1f} s.")

rows = []
for ds in DS:
    g = GEN[ds]
    for lm_name, lm in LM.items():
        for method in ["LogTopK", "NearestFeasible", "DiCE", "CFNOW", "ACFX"]:
            t_int = lm["parse"] + g[method] + lm["explain"]
            for n_conf in (0, 1):
                T = t_int + n_conf * g["retrain_mean"]
                rows.append(dict(dataset=ds, lm=lm_name, method=method, confirm_top=n_conf, interaction_s=T,
                                 break_even_K_candidates=T / g["retrain_mean"],
                                 speedup_vs_K8_sequential=8 * g["retrain_mean"] / T,
                                 speedup_vs_K20_sequential=20 * g["retrain_mean"] / T))
COST = pd.DataFrame(rows)
display(COST.round(2))
save_table(COST.round(3), "R05_cost_break_even")

# amortisation of the log itself: after how many exploratory queries does (log cost + Q*interaction) beat Q*K sequential retrainings?
rows = []
for ds in DS:
    g = GEN[ds]; log_cost = g["hpo_total_gpu_h"] * 3600
    for lm_name, lm in LM.items():
        for method in ["LogTopK", "DiCE", "ACFX"]:
            t_int = lm["parse"] + g[method] + lm["explain"]
            for Kc in (8, 20):
                saving_per_query = Kc * g["retrain_mean"] - (t_int + 1 * g["retrain_mean"])    # incl. one confirmation
                rows.append(dict(dataset=ds, lm=lm_name, method=method, K_alternatives=Kc, saving_per_query_s=saving_per_query,
                                 queries_to_amortise_log=(log_cost / saving_per_query if saving_per_query > 0 else np.inf),
                                 hpo_log_cost_gpu_h=g["hpo_total_gpu_h"]))
AM = pd.DataFrame(rows)
display(AM.round(2))
save_table(AM.round(3), "R05_log_amortisation")
print("NOTE: the HPO log is a by-product of an optimisation that is run anyway; the amortisation table is only "
      "relevant if the log would NOT otherwise exist.")

In [ ]:
# ============================================================
# R5.7) Freeze
# ============================================================
merge_numbers("R05_baseline_density_cost", dict(
    retrieval_aggregate=agg.round(4).to_dict(orient="records"),
    recorded_vs_logtopk=H.groupby("dataset").agg(mean_jaccard=("jaccard_recordedtopk_vs_logtopk", "mean"),
                                                 mean_recorded_gain=("recorded_gain_of_surrogate_rank", "mean")).round(4).reset_index().to_dict(orient="records"),
    density_summary=SUM.round(4).to_dict(orient="records"),
    costs=GEN,
    break_even=COST[(COST.confirm_top == 0)].round(3).to_dict(orient="records"),
))